In [1]:
# ============================================================
# Notebook 26
# 26_fold_rule_sensitivity.ipynb
#
# Purpose:
#   Sensitivity analysis for duplicate overlapping-fold probability handling.
#
# Reviewer issue addressed:
#   Overlapping walk-forward validation/test folds can produce multiple eligible
#   out-of-sample probability rows for the same date. Notebook 25 documented
#   the latest-eligible-fold rule. This notebook tests whether results are
#   sensitive to alternative deterministic aggregation rules:
#
#     1. latest eligible fold
#     2. earliest eligible fold
#     3. average of all eligible out-of-sample probability vectors
#
# Main outputs:
#   outputs/AURORA_TWETF/fold_rule_sensitivity/run_<RUN_ID>/
#     tables/table_S38_fold_rule_sensitivity_design.csv
#     tables/table_S39_fold_rule_probability_sensitivity.csv
#     tables/table_S40_fold_rule_portfolio_sensitivity.csv
#     tables/table_S41_fold_rule_source_composition.csv
#     tables/table_S42_fold_rule_difference_vs_latest.csv
#     diagnostics/fold_rule_selected_probability_map.csv
#     returns/fold_rule_sensitivity_returns.parquet
#     weights/fold_rule_sensitivity_weights.parquet
#     reports/NOTEBOOK26_fold_rule_sensitivity_validation_report.json
#
# Research diagnostics only. Not financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
import re
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive mount skipped or already mounted:", repr(e))

import numpy as np
import pandas as pd

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    print("scipy.optimize unavailable. Optimizer will use fallback allocations.")

# ============================================================
# 0. Paths and global settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
MODELING_DIR = DATA_ROOT / "modeling"
PANEL_DIR = DATA_ROOT / "panels"
RAW_YF_DIR = DATA_ROOT / "raw_yfinance"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
DIAGNOSTIC_GLOBAL_DIR = OUTPUT_ROOT / "diagnostics"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "fold_rule_sensitivity" / f"run_{RUN_ID}"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAG_DIR = RUN_ROOT / "diagnostics"
RETURN_DIR = RUN_ROOT / "returns"
WEIGHT_DIR = RUN_ROOT / "weights"

for d in [
    RUN_ROOT,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    DIAG_DIR,
    RETURN_DIR,
    WEIGHT_DIR,
    TABLE_DIR,
    REPORT_DIR,
    DIAGNOSTIC_GLOBAL_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Evaluation window used in the manuscript.
EVAL_START = pd.Timestamp("2024-11-27")
EVAL_END = pd.Timestamp("2026-03-25")

# AURORA target columns.
TARGET_20D = "TAIEX_regime_fixed_20d"
TARGET_60D = "TAIEX_regime_fixed_60d"
TARGETS = [TARGET_20D, TARGET_60D]
TARGET_HORIZON = {
    TARGET_20D: 20,
    TARGET_60D: 60,
}

CLASS_LABELS = [0, 1, 2, 3, 4]
PROBA_COLS = [f"proba_class_{k}" for k in CLASS_LABELS]

# Allocation universe.
ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "cash"
ASSET_COLS = ETF_UNIVERSE + [CASH_COL]

# AURORA10-UAMV-B-like diagnostic allocation settings.
ANNUALIZATION_DAYS = 252
TRANSACTION_COST_BPS = 10
TRANSACTION_COST_RATE = TRANSACTION_COST_BPS / 10000.0
REBALANCE_CONVENTION = "month_end"

ALPHA_20 = 0.30
ALPHA_60 = 0.70

LAMBDA0 = 10.0
GAMMA_U = 2.5
GAMMA_B = 2.0
RHO = 0.25

MU_LOOKBACK = 63
COV_LOOKBACK = 126
MEAN_SHRINKAGE = 0.60
MOMENTUM_WEIGHT = 0.40
REGIME_TILT_STRENGTH = 0.25

GENERAL_ETF_CAP = 0.45
ETF_00881_CAP = 0.30
CASH_CAP = 0.60

FOLD_RULES = ["latest", "earliest", "average"]

AURORA_WF_RUN_ID = "20260624_031817"
AURORA_WF_ROOT = (
    OUTPUT_ROOT
    / "purged_walk_forward_models"
    / f"run_{AURORA_WF_RUN_ID}"
)

ALLOCATION_INPUT_INDEX_CANDIDATES = [
    AURORA_WF_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv",
    AURORA_WF_ROOT / "NOTEBOOK08_AURORA_ALLOCATION_INPUT_INDEX_PURGED_WF.csv",
    AURORA_WF_ROOT / "NOTEBOOK08_INPUT_INDEX.csv",
]

print("=" * 100)
print("AURORA-TWETF Notebook 26: Fold-rule sensitivity")
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("Evaluation window:", EVAL_START.date(), "to", EVAL_END.date())
print("=" * 100)

# ============================================================
# 1. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    path = Path(path)
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def normalize_name(x):
    return "".join(ch for ch in str(x).lower() if ch.isalnum())

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("+", "plus")
        .replace("=", "_")
    )

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    # Keep as ordinary dataframe; do not set index here.
    # This avoids the pandas ambiguity where "date" is both index level and column label.
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df[df["date"].notna()].copy()
    elif "Date" in df.columns:
        df = df.rename(columns={"Date": "date"})
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df[df["date"].notna()].copy()
    else:
        idx = pd.to_datetime(df.index, errors="coerce")
        if pd.Series(idx).notna().mean() > 0.50:
            df = df.copy()
            df["date"] = idx

    return df.reset_index(drop=True)

def to_datetime_index(df, date_col="date"):
    out = df.copy()

    # Fix ambiguity when date is both index name and column.
    if out.index.name == date_col and date_col in out.columns:
        out = out.reset_index(drop=True)
    elif date_col not in out.columns and out.index.name == date_col:
        out = out.reset_index()

    if date_col in out.columns:
        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
        out = out[out[date_col].notna()].copy()
        out = out.set_index(date_col)
    else:
        out.index = pd.to_datetime(out.index, errors="coerce")
        out = out[out.index.notna()].copy()

    out.index.name = "date"
    return out.sort_index()

def write_table(df, filename_stem, index=False, also_global=True):
    local_path = TABLE_RUN_DIR / f"{filename_stem}.csv"
    df.to_csv(local_path, index=index)
    print("Saved:", local_path)

    global_path = None
    if also_global:
        global_path = TABLE_DIR / f"{filename_stem}.csv"
        df.to_csv(global_path, index=index)
        print("Saved:", global_path)

    return local_path, global_path

def write_rounded_table(df, filename_stem, digits=6, also_global=True):
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(digits)
    return write_table(out, filename_stem, index=False, also_global=also_global)

def parse_fold_number(x):
    s = str(x)
    m = re.search(r"(\d+)", s)
    if m:
        return int(m.group(1))
    return 0

def split_priority_value(x):
    x = str(x).lower()
    if x == "train":
        return 0
    if x in ["validation", "valid", "val"]:
        return 1
    if x in ["test", "strict_test", "oos_test"]:
        return 2
    return 1

# ============================================================
# 2. Load modeling labels and ETF returns
# ============================================================

print("\n" + "=" * 100)
print("Loading modeling labels and ETF return panel")
print("=" * 100)

LABEL_FILE_CANDIDATES = [
    MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet",
    MODELING_DIR / "AURORA_TWETF_features_with_labels.csv",
    MODELING_DIR / "features_with_labels.parquet",
    MODELING_DIR / "features_with_labels.csv",
]

label_path = find_first_existing(LABEL_FILE_CANDIDATES)
if label_path is None:
    raise FileNotFoundError(
        "Could not find modeling label file. Tried:\n"
        + "\n".join(str(p) for p in LABEL_FILE_CANDIDATES)
    )

labels_df = to_datetime_index(read_table_auto(label_path))

for target in TARGETS:
    if target not in labels_df.columns:
        raise ValueError(f"Missing target column {target} in {label_path}")

labels_df = labels_df[TARGETS].dropna().copy()
labels_df[TARGET_20D] = labels_df[TARGET_20D].astype(int)
labels_df[TARGET_60D] = labels_df[TARGET_60D].astype(int)

print("Label file:", label_path)
print("Label shape:", labels_df.shape)
print("Label dates:", labels_df.index.min().date(), "to", labels_df.index.max().date())

ETF_RETURN_PANEL_CANDIDATES = [
    PANEL_DIR / "AURORA_etf_return_panel.parquet",
    PANEL_DIR / "AURORA_TWETF_etf_return_panel.parquet",
    PANEL_DIR / "etf_return_panel.parquet",
    PANEL_DIR / "AURORA_etf_returns.parquet",
    PANEL_DIR / "AURORA_etf_return_panel.csv",
]

def load_etf_returns_from_panel(path):
    df = to_datetime_index(read_table_auto(path))
    rename = {}
    for c in df.columns:
        nc = normalize_name(c)
        if "006208" in nc:
            rename[c] = "006208"
        elif "00692" in nc:
            rename[c] = "00692"
        elif "00881" in nc:
            rename[c] = "00881"
        elif "0050" in nc:
            rename[c] = "0050"
    df = df.rename(columns=rename)
    missing = [c for c in ETF_UNIVERSE if c not in df.columns]
    if missing:
        raise ValueError(f"ETF return panel missing columns: {missing}")
    out = df[ETF_UNIVERSE].apply(pd.to_numeric, errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out.sort_index()

def load_raw_price_for_etf(symbol):
    candidates = [
        RAW_YF_DIR / f"{symbol}_{symbol}_TW.csv",
        RAW_YF_DIR / f"{symbol}.TW.csv",
        RAW_YF_DIR / f"{symbol}.csv",
    ]
    p = find_first_existing(candidates)
    if p is None:
        raise FileNotFoundError(f"Could not find raw yfinance file for {symbol}. Tried: {candidates}")

    df = to_datetime_index(read_table_auto(p))

    desired_adj = f"Adj Close_{symbol}.TW"
    desired_close = f"Close_{symbol}.TW"

    if desired_adj in df.columns:
        col = desired_adj
    elif desired_close in df.columns:
        col = desired_close
    else:
        adj_cols = [c for c in df.columns if "adjclose" in normalize_name(c)]
        close_cols = [c for c in df.columns if "close" in normalize_name(c) and "adj" not in normalize_name(c)]
        if adj_cols:
            col = adj_cols[0]
        elif close_cols:
            col = close_cols[0]
        else:
            raise ValueError(f"No close or adjusted-close column found for {symbol} in {p}")

    s = pd.to_numeric(df[col], errors="coerce").dropna()
    s.name = symbol
    return s.sort_index()

def load_etf_returns_from_raw():
    prices = []
    for sym in ETF_UNIVERSE:
        prices.append(load_raw_price_for_etf(sym))
    price_df = pd.concat(prices, axis=1).sort_index()
    ret_df = price_df.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how="all").fillna(0.0)
    return ret_df[ETF_UNIVERSE].copy()

etf_return_path = find_first_existing(ETF_RETURN_PANEL_CANDIDATES)

if etf_return_path is not None:
    etf_returns = load_etf_returns_from_panel(etf_return_path)
    print("ETF return panel:", etf_return_path)
else:
    etf_returns = load_etf_returns_from_raw()
    etf_return_path = "raw_yfinance_reconstructed"
    print("ETF returns reconstructed from raw_yfinance files")

print("ETF return shape:", etf_returns.shape)
print("ETF return dates:", etf_returns.index.min().date(), "to", etf_returns.index.max().date())

eval_dates_base = labels_df.index.intersection(etf_returns.index).sort_values()
eval_dates_base = eval_dates_base[
    (eval_dates_base >= EVAL_START)
    & (eval_dates_base <= EVAL_END)
]

if len(eval_dates_base) == 0:
    raise ValueError("No aligned label/ETF return dates in evaluation window.")

print(
    "Base aligned evaluation dates:",
    len(eval_dates_base),
    eval_dates_base.min().date(),
    "to",
    eval_dates_base.max().date(),
)

# ============================================================
# 3. Load selected probability files before duplicate handling
# ============================================================

print("\n" + "=" * 100)
print("Loading selected pre-deduplication probability files")
print("=" * 100)

def locate_allocation_input_index():
    p = find_first_existing(ALLOCATION_INPUT_INDEX_CANDIDATES)
    if p is not None:
        return p

    candidates = list((OUTPUT_ROOT / "purged_walk_forward_models").rglob("*INPUT_INDEX*.csv"))
    if candidates:
        ranked = sorted(
            candidates,
            key=lambda x: (
                0 if "NOTEBOOK08" in x.name.upper() else 1,
                len(str(x)),
            )
        )
        return ranked[0]

    return None

input_index_path = locate_allocation_input_index()

if input_index_path is None:
    print("Warning: allocation input index not found. Will search probability files directly.")
    input_index = pd.DataFrame()
else:
    input_index = pd.read_csv(input_index_path)
    print("Allocation input index:", input_index_path)
    print("Index shape:", input_index.shape)
    print("Index columns:", list(input_index.columns))

def rank_probability_file(path, target_col):
    name = str(path).lower()
    score = 0

    if target_col.lower() in name:
        score -= 50
    if "selected" in name:
        score -= 25
    if "probab" in name or "proba" in name:
        score -= 15
    if "validation" in name and "test" in name:
        score -= 10
    if "dedup" in name:
        score += 40
    if "fold_source_map" in name:
        score += 100
    if "matrix" in name:
        score += 80
    if path.suffix.lower() == ".parquet":
        score -= 5

    return score

def find_probability_path_from_index(target_col):
    if input_index.empty:
        return None

    target_rows = input_index.copy()

    possible_target_cols = [
        c for c in target_rows.columns
        if "target" in normalize_name(c)
        or "label" in normalize_name(c)
    ]

    filtered = pd.DataFrame()
    for c in possible_target_cols:
        tmp = target_rows[target_rows[c].astype(str).str.contains(target_col, case=False, regex=False, na=False)].copy()
        if not tmp.empty:
            filtered = tmp
            break

    if filtered.empty:
        horizon = TARGET_HORIZON[target_col]
        possible_horizon_cols = [c for c in target_rows.columns if "horizon" in normalize_name(c)]
        for c in possible_horizon_cols:
            tmp = target_rows[pd.to_numeric(target_rows[c], errors="coerce") == horizon].copy()
            if not tmp.empty:
                filtered = tmp
                break

    if filtered.empty:
        filtered = target_rows.copy()

    probability_cols = [
        c for c in filtered.columns
        if (
            "prob" in normalize_name(c)
            or "proba" in normalize_name(c)
        )
        and (
            "path" in normalize_name(c)
            or "file" in normalize_name(c)
        )
    ]

    found_paths = []
    for _, row in filtered.iterrows():
        for c in probability_cols:
            if pd.notna(row[c]):
                p = Path(str(row[c]))
                if p.exists() and p.suffix.lower() in [".parquet", ".csv"]:
                    found_paths.append(p)

    if found_paths:
        found_paths = sorted(set(found_paths), key=lambda p: rank_probability_file(p, target_col))
        return found_paths[0]

    return None

def find_probability_path_by_search(target_col):
    search_roots = [
        AURORA_WF_ROOT,
        OUTPUT_ROOT / "purged_walk_forward_models",
        OUTPUT_ROOT,
    ]

    candidates = []
    for root in search_roots:
        root = Path(root)
        if root.exists():
            candidates.extend(root.rglob(f"*{target_col}*.parquet"))
            candidates.extend(root.rglob(f"*{target_col}*.csv"))

    candidates = [
        p for p in candidates
        if (
            ("prob" in p.name.lower() or "proba" in p.name.lower())
            and "fold_source_map" not in p.name.lower()
        )
    ]

    if not candidates:
        return None

    candidates = sorted(set(candidates), key=lambda p: rank_probability_file(p, target_col))
    return candidates[0]

def load_probability_file_for_target(target_col):
    p = find_probability_path_from_index(target_col)
    if p is None:
        p = find_probability_path_by_search(target_col)

    if p is None:
        raise FileNotFoundError(f"Could not locate selected probability file for {target_col}")

    df = read_table_auto(p)
    df["source_probability_path"] = str(p)
    return df, p

def standardize_probability_df(df, target_col):
    out = df.copy()

    # CRITICAL FIX:
    # Avoid pandas ambiguity when "date" exists as both index level and column.
    # This can happen after reading probability files with an existing date column
    # while the dataframe index also has name "date".
    if out.index.name == "date" and "date" in out.columns:
        out = out.reset_index(drop=True)
    elif "date" not in out.columns and out.index.name == "date":
        out = out.reset_index()
    elif "date" not in out.columns:
        date_like_cols = [
            c for c in out.columns
            if str(c).lower() in ["date", "datetime", "timestamp", "index", "__index_level_0__"]
        ]
        if date_like_cols:
            out = out.rename(columns={date_like_cols[0]: "date"})
        else:
            raise ValueError(
                f"Probability file for {target_col} has no date column after standardization. "
                f"Columns: {list(out.columns)[:60]}"
            )

    out = out.reset_index(drop=True)
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out[out["date"].notna()].copy()

    # Standardize probability column names if necessary.
    rename = {}
    for c in out.columns:
        nc = normalize_name(c)
        for k in CLASS_LABELS:
            candidates = [
                f"probaclass{k}",
                f"probclass{k}",
                f"pclass{k}",
                f"p{k}",
                f"prob{k}",
            ]
            if nc in candidates:
                rename[c] = f"proba_class_{k}"

    out = out.rename(columns=rename)

    missing = [c for c in PROBA_COLS if c not in out.columns]
    if missing:
        raise ValueError(
            f"Probability file for {target_col} missing probability columns: {missing}. "
            f"Available columns: {list(out.columns)[:60]}"
        )

    for c in PROBA_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    # Normalize probability rows.
    p = out[PROBA_COLS].to_numpy(dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sum = p.sum(axis=1, keepdims=True)
    zero = row_sum[:, 0] <= 0

    if zero.any():
        p[zero, :] = 1.0 / len(CLASS_LABELS)
        row_sum = p.sum(axis=1, keepdims=True)

    p = p / row_sum

    for j, c in enumerate(PROBA_COLS):
        out[c] = p[:, j]

    if "target_col" not in out.columns:
        out["target_col"] = target_col

    if "horizon" not in out.columns:
        out["horizon"] = TARGET_HORIZON[target_col]

    if "fold_id" not in out.columns:
        out["fold_id"] = "unknown"

    if "split" not in out.columns:
        out["split"] = "unknown"

    out["fold_number"] = out["fold_id"].map(parse_fold_number).astype(int)

    out["split"] = out["split"].astype(str).str.lower()
    out["split"] = out["split"].replace({
        "valid": "validation",
        "val": "validation",
        "strict_test": "test",
        "oos_test": "test",
    })

    out["split_priority"] = out["split"].map(split_priority_value).astype(int)

    eligible_splits = ["validation", "test", "unknown"]
    out = out[out["split"].isin(eligible_splits)].copy()

    out = out.sort_values(["date", "split_priority", "fold_number"]).reset_index(drop=True)

    return out

probability_raw = {}
probability_paths = {}

for target in TARGETS:
    df, path = load_probability_file_for_target(target)
    df = standardize_probability_df(df, target)
    probability_raw[target] = df
    probability_paths[target] = path

    print("\nTarget:", target)
    print("Probability path:", path)
    print("Shape:", df.shape)
    print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
    print("Split counts:", df["split"].value_counts(dropna=False).to_dict())
    print("Fold counts:", df["fold_id"].astype(str).value_counts(dropna=False).to_dict())

# ============================================================
# 4. Fold-rule aggregation
# ============================================================

print("\n" + "=" * 100)
print("Applying duplicate probability aggregation rules")
print("=" * 100)

def eligible_count_map(df):
    return df.groupby("date").size().to_dict()

def aggregate_probability_rows(df, rule, target_col):
    df2 = df.copy()
    df2 = df2.reset_index(drop=True)
    df2["date"] = pd.to_datetime(df2["date"])
    df2 = df2.sort_values(["date", "split_priority", "fold_number"]).reset_index(drop=True)

    count_map = eligible_count_map(df2)

    if rule == "latest":
        selected = df2.drop_duplicates(subset=["date"], keep="last").copy()
        selected["aggregation_rule"] = "latest"
        selected["n_eligible_rows_for_date"] = selected["date"].map(count_map).astype(int)
        selected["source_summary"] = selected["split"].astype(str) + ":" + selected["fold_id"].astype(str)
        return selected.reset_index(drop=True)

    if rule == "earliest":
        selected = df2.drop_duplicates(subset=["date"], keep="first").copy()
        selected["aggregation_rule"] = "earliest"
        selected["n_eligible_rows_for_date"] = selected["date"].map(count_map).astype(int)
        selected["source_summary"] = selected["split"].astype(str) + ":" + selected["fold_id"].astype(str)
        return selected.reset_index(drop=True)

    if rule == "average":
        rows = []
        for dt, g in df2.groupby("date"):
            p_mean = g[PROBA_COLS].mean(axis=0)
            if p_mean.sum() > 0:
                p_mean = p_mean / p_mean.sum()
            else:
                p_mean = pd.Series(1.0 / len(PROBA_COLS), index=PROBA_COLS)

            split_counts = g["split"].value_counts().to_dict()
            fold_counts = g["fold_id"].astype(str).value_counts().to_dict()

            row = {
                "date": dt,
                "target_col": target_col,
                "horizon": TARGET_HORIZON[target_col],
                "fold_id": "average",
                "fold_number": np.nan,
                "split": "average",
                "split_priority": np.nan,
                "aggregation_rule": "average",
                "n_eligible_rows_for_date": int(len(g)),
                "source_summary": (
                    "splits="
                    + json.dumps(split_counts, sort_keys=True)
                    + "; folds="
                    + json.dumps(fold_counts, sort_keys=True)
                ),
            }
            for c in PROBA_COLS:
                row[c] = float(p_mean[c])
            rows.append(row)

        return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

    raise ValueError(f"Unknown rule: {rule}")

aggregated = {}
for rule in FOLD_RULES:
    aggregated[rule] = {}
    for target in TARGETS:
        agg = aggregate_probability_rows(probability_raw[target], rule, target)
        aggregated[rule][target] = agg

        print(
            f"{rule} | {target}: n_dates={agg['date'].nunique()}, "
            f"range={agg['date'].min().date()} to {agg['date'].max().date()}"
        )

# ============================================================
# 5. Probability diagnostics
# ============================================================

def normalize_proba_array(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0
    row_sum = p.sum(axis=1, keepdims=True)
    zero = row_sum[:, 0] <= 0
    if zero.any():
        p[zero, :] = 1.0 / p.shape[1]
        row_sum = p.sum(axis=1, keepdims=True)
    return p / row_sum

def onehot(labels):
    labels = pd.Series(labels).astype(int).values
    out = np.zeros((len(labels), len(CLASS_LABELS)), dtype=float)
    for i, y in enumerate(labels):
        if y in CLASS_LABELS:
            out[i, CLASS_LABELS.index(int(y))] = 1.0
        else:
            out[i, :] = 1.0 / len(CLASS_LABELS)
    return out

def multiclass_brier(y_true, proba):
    y_true = pd.Series(y_true).astype(int).values
    proba = normalize_proba_array(proba)
    y_oh = onehot(y_true)
    return float(np.mean(np.sum((proba - y_oh) ** 2, axis=1)))

def hard_accuracy(y_true, proba):
    y_true = pd.Series(y_true).astype(int).values
    pred = np.asarray(CLASS_LABELS)[np.argmax(normalize_proba_array(proba), axis=1)]
    return float(np.mean(pred == y_true))

def probability_features_from_p20_p60(index, p20, p60, alpha20=ALPHA_20, alpha60=ALPHA_60):
    p20 = normalize_proba_array(p20)
    p60 = normalize_proba_array(p60)

    s = alpha20 + alpha60
    if s <= 0:
        alpha20, alpha60 = 0.5, 0.5
    else:
        alpha20, alpha60 = alpha20 / s, alpha60 / s

    p = alpha20 * p20 + alpha60 * p60
    p = normalize_proba_array(p)

    clipped = np.clip(p, 1e-12, 1.0)
    entropy = -np.sum(clipped * np.log(clipped), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))

    class_values = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = p @ class_values
    ordinal_variance = p @ (class_values ** 2) - expected_class ** 2

    out = pd.DataFrame(index=pd.DatetimeIndex(index).sort_values())
    out.index.name = "date"
    for i, cls in enumerate(CLASS_LABELS):
        out[f"proba_class_{cls}"] = p[:, i]

    out["expected_class"] = expected_class
    out["entropy"] = entropy
    out["normalized_entropy"] = normalized_entropy
    out["confidence_score"] = 1.0 - normalized_entropy
    out["ordinal_variance"] = ordinal_variance
    out["p_bearish"] = p[:, 0] + p[:, 1]
    out["p_bullish"] = p[:, 3] + p[:, 4]
    out["risk_on_score"] = (expected_class - 2.0) / 2.0
    out["lambda_dynamic"] = LAMBDA0 * (1.0 + GAMMA_U * out["normalized_entropy"] + GAMMA_B * out["p_bearish"])

    return out

probability_sensitivity_rows = []
source_composition_rows = []
probability_map_rows = []

rule_features = {}
rule_aligned_data = {}

for rule in FOLD_RULES:
    p20_df = aggregated[rule][TARGET_20D].copy()
    p60_df = aggregated[rule][TARGET_60D].copy()

    p20_idx = pd.DatetimeIndex(p20_df["date"])
    p60_idx = pd.DatetimeIndex(p60_df["date"])

    common = (
        p20_idx
        .intersection(p60_idx)
        .intersection(labels_df.index)
        .intersection(etf_returns.index)
        .sort_values()
    )
    common = common[(common >= EVAL_START) & (common <= EVAL_END)]

    p20a = p20_df.set_index("date").loc[common].sort_index()
    p60a = p60_df.set_index("date").loc[common].sort_index()

    y20 = labels_df.loc[common, TARGET_20D].astype(int)
    y60 = labels_df.loc[common, TARGET_60D].astype(int)

    p20 = p20a[PROBA_COLS].to_numpy(dtype=float)
    p60 = p60a[PROBA_COLS].to_numpy(dtype=float)

    features = probability_features_from_p20_p60(common, p20, p60)
    rule_features[rule] = features
    rule_aligned_data[rule] = {
        "dates": common,
        "p20": p20a,
        "p60": p60a,
        "y20": y20,
        "y60": y60,
    }

    probability_sensitivity_rows.append({
        "aggregation_rule": rule,
        "n_dates": int(len(common)),
        "start_date": common.min() if len(common) else pd.NaT,
        "end_date": common.max() if len(common) else pd.NaT,
        "brier_20d": multiclass_brier(y20, p20),
        "brier_60d": multiclass_brier(y60, p60),
        "weighted_brier_30_70": ALPHA_20 * multiclass_brier(y20, p20) + ALPHA_60 * multiclass_brier(y60, p60),
        "hard_accuracy_20d": hard_accuracy(y20, p20),
        "hard_accuracy_60d": hard_accuracy(y60, p60),
        "avg_entropy": float(features["normalized_entropy"].mean()),
        "median_entropy": float(features["normalized_entropy"].median()),
        "avg_bearish_probability": float(features["p_bearish"].mean()),
        "median_bearish_probability": float(features["p_bearish"].median()),
        "avg_dynamic_lambda": float(features["lambda_dynamic"].mean()),
        "median_dynamic_lambda": float(features["lambda_dynamic"].median()),
    })

    for target, df_aligned in [(TARGET_20D, p20a), (TARGET_60D, p60a)]:
        raw = probability_raw[target].copy()
        raw_eval = raw[
            (raw["date"] >= EVAL_START)
            & (raw["date"] <= EVAL_END)
        ].copy()

        duplicate_counts = raw_eval.groupby("date").size()

        source_composition_rows.append({
            "aggregation_rule": rule,
            "target_col": target,
            "horizon": TARGET_HORIZON[target],
            "aligned_dates": int(len(df_aligned)),
            "eligible_rows_before_rule_in_eval_window": int(len(raw_eval)),
            "dates_with_multiple_eligible_rows_before_rule": int((duplicate_counts > 1).sum()),
            "max_eligible_rows_per_date_before_rule": int(duplicate_counts.max()) if len(duplicate_counts) else 0,
            "mean_eligible_rows_per_selected_date": float(df_aligned["n_eligible_rows_for_date"].mean()),
            "selected_split_counts": json.dumps(df_aligned["split"].astype(str).value_counts().to_dict(), sort_keys=True),
            "selected_fold_counts": json.dumps(df_aligned["fold_id"].astype(str).value_counts().to_dict(), sort_keys=True),
            "all_selected_dates_test_only": bool((df_aligned["split"].astype(str) == "test").all()) if rule in ["latest", "earliest"] else False,
            "note": (
                "For average rule, selected_split_counts refers to averaged rows and detailed source mix is in source_summary."
                if rule == "average"
                else "For latest/earliest rules, selected rows have one source row per date."
            ),
        })

    for dt in common:
        row = {
            "date": dt,
            "aggregation_rule": rule,
            "p20_source_summary": p20a.loc[dt, "source_summary"],
            "p60_source_summary": p60a.loc[dt, "source_summary"],
            "p20_n_eligible": int(p20a.loc[dt, "n_eligible_rows_for_date"]),
            "p60_n_eligible": int(p60a.loc[dt, "n_eligible_rows_for_date"]),
            "p20_split": str(p20a.loc[dt, "split"]),
            "p60_split": str(p60a.loc[dt, "split"]),
            "p20_fold_id": str(p20a.loc[dt, "fold_id"]),
            "p60_fold_id": str(p60a.loc[dt, "fold_id"]),
        }
        for c in PROBA_COLS:
            row[f"p20_{c}"] = float(p20a.loc[dt, c])
            row[f"p60_{c}"] = float(p60a.loc[dt, c])
            row[f"blend_{c}"] = float(features.loc[dt, c])
        row["blend_entropy"] = float(features.loc[dt, "normalized_entropy"])
        row["blend_bearish_probability"] = float(features.loc[dt, "p_bearish"])
        row["blend_lambda_dynamic"] = float(features.loc[dt, "lambda_dynamic"])
        probability_map_rows.append(row)

probability_sensitivity_df = pd.DataFrame(probability_sensitivity_rows)
source_composition_df = pd.DataFrame(source_composition_rows)
probability_map_df = pd.DataFrame(probability_map_rows)

print("\nProbability sensitivity:")
print(probability_sensitivity_df.round(6).to_string(index=False))

print("\nSource composition:")
print(source_composition_df.to_string(index=False))

# ============================================================
# 6. AURORA-compatible optimizer and portfolio backtest
# ============================================================

def make_rebalance_dates(index):
    index = pd.DatetimeIndex(index).sort_values()
    ser = pd.Series(index=index, data=index)
    if REBALANCE_CONVENTION == "month_start":
        return pd.DatetimeIndex(ser.groupby(index.to_period("M")).min().values).sort_values()
    return pd.DatetimeIndex(ser.groupby(index.to_period("M")).max().values).sort_values()

def cap_vector():
    return np.array([GENERAL_ETF_CAP, GENERAL_ETF_CAP, GENERAL_ETF_CAP, ETF_00881_CAP, CASH_CAP], dtype=float)

def cap_and_normalize_weights(w):
    w = np.asarray(w, dtype=float).copy()
    caps = cap_vector()

    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w = np.maximum(w, 0.0)
    w = np.minimum(w, caps)

    for _ in range(50):
        gap = 1.0 - w.sum()
        if abs(gap) < 1e-10:
            break

        if gap > 0:
            capacity = caps - w
            capacity[capacity < 0] = 0.0
            if capacity.sum() <= 1e-12:
                break
            w += gap * capacity / capacity.sum()
            w = np.minimum(w, caps)
        else:
            positive = w > 0
            if positive.sum() == 0:
                break
            w[positive] += gap * w[positive] / w[positive].sum()
            w = np.maximum(w, 0.0)

    if abs(w.sum() - 1.0) > 1e-6:
        w = np.array([0.10, 0.10, 0.10, 0.10, 0.60], dtype=float)

    return w / w.sum()

def initial_weight():
    return np.array([0.10, 0.10, 0.10, 0.10, 0.60], dtype=float)

def regime_tilt_vector(features_row):
    risk_on = float(features_row.get("risk_on_score", 0.0))
    p_bearish = float(features_row.get("p_bearish", 0.0))
    p_bullish = float(features_row.get("p_bullish", 0.0))

    tilt = pd.Series(0.0, index=ETF_UNIVERSE)
    tilt["0050"] += 0.15 * risk_on
    tilt["006208"] += 0.15 * risk_on
    tilt["00692"] += 0.05 * risk_on + 0.10 * p_bearish
    tilt["00881"] += 0.35 * risk_on + 0.15 * p_bullish - 0.20 * p_bearish
    return tilt.values

def estimate_moments_for_date(etf_returns, dt, features_row, use_regime_tilt=True):
    hist = etf_returns.loc[etf_returns.index < dt, ETF_UNIVERSE].dropna(how="all").fillna(0.0)

    if len(hist) < max(MU_LOOKBACK, COV_LOOKBACK):
        return None, None

    r_mu = hist.tail(MU_LOOKBACK)
    r_cov = hist.tail(COV_LOOKBACK)

    mean_daily = r_mu.mean().values
    cumulative = (1.0 + r_mu).prod().values - 1.0
    momentum_daily = cumulative / max(len(r_mu), 1)

    risky_mu = (1.0 - MEAN_SHRINKAGE) * (
        (1.0 - MOMENTUM_WEIGHT) * mean_daily
        + MOMENTUM_WEIGHT * momentum_daily
    )

    if use_regime_tilt:
        realized_vol = r_cov.std().replace(0.0, np.nan)
        vol_scale = realized_vol.median()
        if not np.isfinite(vol_scale) or vol_scale <= 0:
            vol_scale = 0.01

        risky_mu = risky_mu + REGIME_TILT_STRENGTH * regime_tilt_vector(features_row) * vol_scale / ANNUALIZATION_DAYS

    cov = r_cov.cov().values
    cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)

    avg_var = np.nanmean(np.diag(cov))
    if not np.isfinite(avg_var) or avg_var <= 0:
        avg_var = 1e-6

    cov = cov + np.eye(len(ETF_UNIVERSE)) * max(1e-10, avg_var * 0.10)

    mu = np.concatenate([risky_mu, [0.0]])

    sigma = np.zeros((5, 5), dtype=float)
    sigma[:4, :4] = cov
    sigma[4, 4] = 1e-12

    return mu, sigma

def optimize_weight(mu, sigma, lam, prev_w):
    prev_w = cap_and_normalize_weights(prev_w)
    caps = cap_vector()
    bounds = [(0.0, caps[i]) for i in range(5)]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    if mu is None or sigma is None or not SCIPY_AVAILABLE:
        return prev_w, {
            "solver_success": False,
            "fallback_used": True,
            "message": "missing_moments_or_scipy",
            "objective_value": np.nan,
            "equality_residual": abs(prev_w.sum() - 1.0),
            "max_bound_violation": 0.0,
        }

    def objective(w):
        w = np.asarray(w, dtype=float)
        return -(
            float(mu @ w)
            - float(lam) * float(w.T @ sigma @ w)
            - float(RHO) * float(np.sum((w - prev_w) ** 2))
        )

    x0 = prev_w.copy()

    try:
        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 300, "ftol": 1e-10, "disp": False},
        )

        if res.success and np.all(np.isfinite(res.x)):
            w = cap_and_normalize_weights(res.x)
            success = True
            fallback = False
            msg = str(res.message)
            obj = -float(res.fun)
        else:
            w = prev_w.copy()
            success = False
            fallback = True
            msg = str(getattr(res, "message", "optimizer_failed"))
            obj = np.nan
    except Exception as e:
        w = prev_w.copy()
        success = False
        fallback = True
        msg = repr(e)[:160]
        obj = np.nan

    eq_resid = abs(float(w.sum() - 1.0))
    max_bound_viol = float(max(0.0, np.max(w - caps), -np.min(w)))

    return w, {
        "solver_success": bool(success),
        "fallback_used": bool(fallback),
        "message": msg,
        "objective_value": obj,
        "equality_residual": eq_resid,
        "max_bound_violation": max_bound_viol,
    }

def performance_metrics_from_returns(r):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)

    if n == 0:
        return {}

    equity = (1.0 + r).cumprod()
    dd = equity / equity.cummax() - 1.0

    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) else np.nan
    sharpe = float((r.mean() / daily_vol) * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float((r.mean() / downside_vol) * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    max_dd = float(dd.min())
    calmar = float(annual_return / abs(max_dd)) if max_dd < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "avg_daily_return": float(r.mean()),
        "daily_volatility": daily_vol,
        "hit_rate": float((r > 0).mean()),
        "final_equity": float(equity.iloc[-1]),
    }

def run_aurora_like_backtest(policy_name, etf_returns, features):
    dates = pd.DatetimeIndex(features.index).intersection(etf_returns.index).sort_values()
    dates = dates[(dates >= EVAL_START) & (dates <= EVAL_END)]

    rebalance_dates = set(make_rebalance_dates(dates))
    current_w = initial_weight()

    ret_rows = []
    weight_rows = []
    diag_rows = []

    for dt in dates:
        tc = 0.0
        turnover = 0.0
        is_rebalance = dt in rebalance_dates

        frow = features.loc[dt]

        if is_rebalance:
            mu, sigma = estimate_moments_for_date(
                etf_returns=etf_returns,
                dt=dt,
                features_row=frow,
                use_regime_tilt=True,
            )

            lam = float(frow["lambda_dynamic"])
            new_w, opt_diag = optimize_weight(mu, sigma, lam, current_w)

            turnover = float(np.sum(np.abs(new_w - current_w)))
            tc = turnover * TRANSACTION_COST_RATE
            current_w = new_w.copy()

            diag_rows.append({
                "date": dt,
                "policy_name": policy_name,
                "lambda_dynamic": lam,
                "is_rebalance": True,
                "turnover": turnover,
                "transaction_cost": tc,
                "cash_weight": float(current_w[4]),
                "at_cash_cap": float(current_w[4] >= CASH_CAP - 1e-6),
                "avg_equity_exposure": float(current_w[:4].sum()),
                "expected_class": float(frow["expected_class"]),
                "normalized_entropy": float(frow["normalized_entropy"]),
                "p_bearish": float(frow["p_bearish"]),
                **opt_diag,
            })

        asset_ret = etf_returns.loc[dt, ETF_UNIVERSE].fillna(0.0).values
        gross_return = float(np.dot(current_w[:4], asset_ret))
        net_return = gross_return - tc

        ret_rows.append({
            "date": dt,
            "policy_name": policy_name,
            "gross_return": gross_return,
            "transaction_cost": tc,
            "net_return": net_return,
        })

        weight_rows.append({
            "date": dt,
            "policy_name": policy_name,
            "0050": float(current_w[0]),
            "006208": float(current_w[1]),
            "00692": float(current_w[2]),
            "00881": float(current_w[3]),
            "cash": float(current_w[4]),
            "is_rebalance": bool(is_rebalance),
        })

        # Drift weights after returns; cash return is zero.
        post_values = current_w.copy()
        post_values[:4] = post_values[:4] * (1.0 + asset_ret)
        post_values[4] = post_values[4]

        if post_values.sum() > 0:
            current_w = post_values / post_values.sum()
        else:
            current_w = initial_weight()

    returns = pd.DataFrame(ret_rows).set_index("date").sort_index()
    weights = pd.DataFrame(weight_rows).set_index("date").sort_index()
    diagnostics = pd.DataFrame(diag_rows).set_index("date").sort_index() if diag_rows else pd.DataFrame()

    returns["equity"] = (1.0 + returns["net_return"]).cumprod()
    returns["drawdown"] = returns["equity"] / returns["equity"].cummax() - 1.0

    return returns, weights, diagnostics

# ============================================================
# 7. Run portfolio sensitivity
# ============================================================

print("\n" + "=" * 100)
print("Running AURORA-compatible fold-rule portfolio sensitivity")
print("=" * 100)

portfolio_rows = []
all_returns = []
all_weights = []
all_optimizer_diag = []

for rule in FOLD_RULES:
    policy_name = f"AURORA_fold_rule_{rule}"
    features = rule_features[rule]

    ret_df, w_df, d_df = run_aurora_like_backtest(
        policy_name=policy_name,
        etf_returns=etf_returns,
        features=features,
    )

    all_returns.append(ret_df.reset_index())
    all_weights.append(w_df.reset_index())

    if d_df is not None and not d_df.empty:
        d_tmp = d_df.reset_index()
        d_tmp["aggregation_rule"] = rule
        all_optimizer_diag.append(d_tmp)

    perf = performance_metrics_from_returns(ret_df["net_return"])

    prob_row = probability_sensitivity_df[
        probability_sensitivity_df["aggregation_rule"] == rule
    ].iloc[0].to_dict()

    perf.update({
        "aggregation_rule": rule,
        "policy_name": policy_name,
        "weighted_brier_30_70": prob_row["weighted_brier_30_70"],
        "brier_20d": prob_row["brier_20d"],
        "brier_60d": prob_row["brier_60d"],
        "avg_entropy": prob_row["avg_entropy"],
        "avg_bearish_probability": prob_row["avg_bearish_probability"],
        "avg_dynamic_lambda": prob_row["avg_dynamic_lambda"],
        "avg_cash": float(w_df["cash"].mean()),
        "median_cash": float(w_df["cash"].median()),
        "cash_cap_binding_frequency": float((w_df["cash"] >= CASH_CAP - 1e-6).mean()),
        "avg_equity_exposure": float(w_df[ETF_UNIVERSE].sum(axis=1).mean()),
        "total_turnover": float(d_df["turnover"].sum()) if d_df is not None and not d_df.empty else np.nan,
        "rebalance_count": int(len(d_df)) if d_df is not None else 0,
        "solver_success_rate": float(d_df["solver_success"].mean()) if d_df is not None and not d_df.empty else np.nan,
        "fallback_rate": float(d_df["fallback_used"].mean()) if d_df is not None and not d_df.empty else np.nan,
    })
    portfolio_rows.append(perf)

    print(
        f"{rule}: TR={perf['total_return']:.6f}, "
        f"Sharpe={perf['sharpe']:.6f}, Sortino={perf['sortino']:.6f}, "
        f"MDD={perf['max_drawdown']:.6f}, avg cash={perf['avg_cash']:.6f}, "
        f"cap={perf['cash_cap_binding_frequency']:.6f}"
    )

portfolio_sensitivity_df = pd.DataFrame(portfolio_rows)

base = portfolio_sensitivity_df[
    portfolio_sensitivity_df["aggregation_rule"] == "latest"
].iloc[0]

difference_rows = []
diff_metrics = [
    "weighted_brier_30_70",
    "brier_20d",
    "brier_60d",
    "avg_entropy",
    "avg_bearish_probability",
    "avg_dynamic_lambda",
    "total_return",
    "sharpe",
    "sortino",
    "max_drawdown",
    "avg_cash",
    "cash_cap_binding_frequency",
    "avg_equity_exposure",
    "total_turnover",
]

for _, row in portfolio_sensitivity_df.iterrows():
    rule = row["aggregation_rule"]
    out = {"aggregation_rule": rule, "comparison": f"{rule}_minus_latest"}
    for m in diff_metrics:
        out[f"delta_{m}"] = float(row[m] - base[m]) if pd.notna(row[m]) and pd.notna(base[m]) else np.nan
    difference_rows.append(out)

difference_vs_latest_df = pd.DataFrame(difference_rows)

print("\nPortfolio sensitivity:")
print(portfolio_sensitivity_df.round(6).to_string(index=False))

print("\nDifferences vs latest:")
print(difference_vs_latest_df.round(6).to_string(index=False))

# ============================================================
# 8. Manuscript-ready tables
# ============================================================

print("\n" + "=" * 100)
print("Building manuscript-ready tables")
print("=" * 100)

table_S38 = pd.DataFrame([
    {
        "component": "Objective",
        "specification": "Test sensitivity to duplicate overlapping-fold probability aggregation.",
        "interpretation": "Evaluates whether final probabilities and AURORA-compatible allocation diagnostics depend on the latest-eligible-fold rule.",
    },
    {
        "component": "Latest eligible fold",
        "specification": "Sort by date, split priority, and fold number; retain the final eligible out-of-sample row per date.",
        "interpretation": "Matches the documented primary probability-retention rule.",
    },
    {
        "component": "Earliest eligible fold",
        "specification": "Sort by date, split priority, and fold number; retain the first eligible out-of-sample row per date.",
        "interpretation": "Tests sensitivity to using the earliest available eligible prediction for each date.",
    },
    {
        "component": "Average eligible probabilities",
        "specification": "Average all eligible validation/test probability vectors for the same date separately for each horizon.",
        "interpretation": "Tests sensitivity to ensemble-style aggregation across overlapping eligible folds.",
    },
    {
        "component": "Horizon treatment",
        "specification": "Aggregation is applied separately to the 20-day and 60-day probability files before common-date alignment.",
        "interpretation": "Preserves the horizon-specific probability construction.",
    },
    {
        "component": "Portfolio diagnostic",
        "specification": "Each aggregated probability sequence is passed through the same AURORA-compatible diagnostic allocation layer.",
        "interpretation": "Reports probability sensitivity and downstream portfolio sensitivity under fixed allocation settings.",
    },
    {
        "component": "Claim use",
        "specification": "Diagnostic robustness check.",
        "interpretation": "Assesses whether the main mechanism interpretation is sensitive to overlapping-fold aggregation choices.",
    },
])

table_S39 = probability_sensitivity_df[
    [
        "aggregation_rule",
        "n_dates",
        "brier_20d",
        "brier_60d",
        "weighted_brier_30_70",
        "avg_entropy",
        "avg_bearish_probability",
        "avg_dynamic_lambda",
    ]
].copy()

table_S40 = portfolio_sensitivity_df[
    [
        "aggregation_rule",
        "n_days",
        "total_return",
        "sharpe",
        "sortino",
        "max_drawdown",
        "avg_cash",
        "cash_cap_binding_frequency",
        "total_turnover",
    ]
].copy()

table_S41 = source_composition_df.copy()

table_S42 = difference_vs_latest_df[
    [
        "aggregation_rule",
        "comparison",
        "delta_weighted_brier_30_70",
        "delta_total_return",
        "delta_sharpe",
        "delta_sortino",
        "delta_max_drawdown",
        "delta_avg_cash",
        "delta_cash_cap_binding_frequency",
    ]
].copy()

write_table(table_S38, "table_S38_fold_rule_sensitivity_design")
write_rounded_table(table_S39, "table_S39_fold_rule_probability_sensitivity")
write_rounded_table(table_S40, "table_S40_fold_rule_portfolio_sensitivity")
write_table(table_S41, "table_S41_fold_rule_source_composition")
write_rounded_table(table_S42, "table_S42_fold_rule_difference_vs_latest")

# Full diagnostics.
probability_map_df.to_csv(DIAG_DIR / "fold_rule_selected_probability_map.csv", index=False)
probability_map_df.to_csv(DIAGNOSTIC_GLOBAL_DIR / "fold_rule_selected_probability_map.csv", index=False)

probability_sensitivity_df.to_csv(DIAG_DIR / "fold_rule_probability_sensitivity_full.csv", index=False)
portfolio_sensitivity_df.to_csv(DIAG_DIR / "fold_rule_portfolio_sensitivity_full.csv", index=False)
source_composition_df.to_csv(DIAG_DIR / "fold_rule_source_composition_full.csv", index=False)
difference_vs_latest_df.to_csv(DIAG_DIR / "fold_rule_difference_vs_latest_full.csv", index=False)

all_returns_df = pd.concat(all_returns, axis=0, ignore_index=True)
all_weights_df = pd.concat(all_weights, axis=0, ignore_index=True)
all_optimizer_diag_df = pd.concat(all_optimizer_diag, axis=0, ignore_index=True) if all_optimizer_diag else pd.DataFrame()

all_returns_df.to_parquet(RETURN_DIR / "fold_rule_sensitivity_returns.parquet", index=False)
all_returns_df.to_csv(RETURN_DIR / "fold_rule_sensitivity_returns.csv", index=False)

all_weights_df.to_parquet(WEIGHT_DIR / "fold_rule_sensitivity_weights.parquet", index=False)
all_weights_df.to_csv(WEIGHT_DIR / "fold_rule_sensitivity_weights.csv", index=False)

all_optimizer_diag_df.to_csv(DIAG_DIR / "fold_rule_optimizer_diagnostics.csv", index=False)

# Global copies.
all_returns_df.to_csv(DIAGNOSTIC_GLOBAL_DIR / "fold_rule_sensitivity_returns.csv", index=False)
all_weights_df.to_csv(DIAGNOSTIC_GLOBAL_DIR / "fold_rule_sensitivity_weights.csv", index=False)

# ============================================================
# 9. Interpretation helper
# ============================================================

interpret_rows = []

for _, row in difference_vs_latest_df.iterrows():
    rule = row["aggregation_rule"]
    if rule == "latest":
        continue

    interpret_rows.append({
        "aggregation_rule": rule,
        "finding": "Difference relative to latest eligible fold",
        "evidence": (
            f"Δ weighted Brier={row['delta_weighted_brier_30_70']:.6f}; "
            f"Δ total return={row['delta_total_return']:.6f}; "
            f"Δ Sharpe={row['delta_sharpe']:.6f}; "
            f"Δ Sortino={row['delta_sortino']:.6f}; "
            f"Δ max drawdown={row['delta_max_drawdown']:.6f}; "
            f"Δ avg cash={row['delta_avg_cash']:.6f}; "
            f"Δ cash-cap binding={row['delta_cash_cap_binding_frequency']:.6f}."
        ),
        "interpretation": (
            "Small differences support robustness of the fold-retention choice; "
            "large differences indicate that overlapping-fold aggregation materially affects probability or allocation diagnostics."
        ),
    })

latest_source = source_composition_df[source_composition_df["aggregation_rule"] == "latest"].copy()
for _, row in latest_source.iterrows():
    interpret_rows.append({
        "aggregation_rule": "latest",
        "finding": f"Final selected-source composition for {row['target_col']}",
        "evidence": (
            f"selected_split_counts={row['selected_split_counts']}; "
            f"selected_fold_counts={row['selected_fold_counts']}; "
            f"all_selected_dates_test_only={row['all_selected_dates_test_only']}."
        ),
        "interpretation": (
            "This row determines whether the final 319-date evaluation probability sequence is drawn entirely from test rows "
            "or includes validation-sourced rows under the documented latest-eligible-fold rule."
        ),
    })

interpret_df = pd.DataFrame(interpret_rows)
write_table(interpret_df, "notebook26_fold_rule_sensitivity_interpretation_helper")

# ============================================================
# 10. Validation report and manifest
# ============================================================

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "26_fold_rule_sensitivity.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Sensitivity analysis for overlapping-fold duplicate probability aggregation. "
        "Compares latest eligible fold, earliest eligible fold, and average of all eligible out-of-sample probability vectors."
    ),
    "evaluation_window": {
        "start": str(EVAL_START.date()),
        "end": str(EVAL_END.date()),
        "base_aligned_label_return_dates": int(len(eval_dates_base)),
    },
    "inputs": {
        "label_file": str(label_path),
        "etf_return_source": str(etf_return_path),
        "allocation_input_index": str(input_index_path) if input_index_path is not None else None,
        "probability_paths": {k: str(v) for k, v in probability_paths.items()},
    },
    "fold_rules": {
        "latest": "Sort by date, split priority, fold number; keep last eligible row per date.",
        "earliest": "Sort by date, split priority, fold number; keep first eligible row per date.",
        "average": "Average all eligible out-of-sample probability rows per date.",
    },
    "allocation_settings": {
        "alpha_20": ALPHA_20,
        "alpha_60": ALPHA_60,
        "lambda0": LAMBDA0,
        "gamma_U": GAMMA_U,
        "gamma_B": GAMMA_B,
        "rho": RHO,
        "mu_lookback": MU_LOOKBACK,
        "cov_lookback": COV_LOOKBACK,
        "mean_shrinkage": MEAN_SHRINKAGE,
        "momentum_weight": MOMENTUM_WEIGHT,
        "regime_tilt_strength": REGIME_TILT_STRENGTH,
        "cash_cap": CASH_CAP,
        "general_etf_cap": GENERAL_ETF_CAP,
        "etf_00881_cap": ETF_00881_CAP,
        "transaction_cost_bps": TRANSACTION_COST_BPS,
        "rebalance_convention": REBALANCE_CONVENTION,
        "scipy_available": SCIPY_AVAILABLE,
    },
    "main_results_preview": {
        "probability_sensitivity": probability_sensitivity_df.to_dict(orient="records"),
        "portfolio_sensitivity": portfolio_sensitivity_df.to_dict(orient="records"),
        "difference_vs_latest": difference_vs_latest_df.to_dict(orient="records"),
    },
    "outputs": {
        "run_root": str(RUN_ROOT),
        "table_S38": str(TABLE_RUN_DIR / "table_S38_fold_rule_sensitivity_design.csv"),
        "table_S39": str(TABLE_RUN_DIR / "table_S39_fold_rule_probability_sensitivity.csv"),
        "table_S40": str(TABLE_RUN_DIR / "table_S40_fold_rule_portfolio_sensitivity.csv"),
        "table_S41": str(TABLE_RUN_DIR / "table_S41_fold_rule_source_composition.csv"),
        "table_S42": str(TABLE_RUN_DIR / "table_S42_fold_rule_difference_vs_latest.csv"),
        "probability_map": str(DIAG_DIR / "fold_rule_selected_probability_map.csv"),
        "returns": str(RETURN_DIR / "fold_rule_sensitivity_returns.parquet"),
        "weights": str(WEIGHT_DIR / "fold_rule_sensitivity_weights.parquet"),
        "optimizer_diagnostics": str(DIAG_DIR / "fold_rule_optimizer_diagnostics.csv"),
    },
    "educational_note": "Research diagnostics only; not personalized financial advice.",
}

validation_path = REPORT_RUN_DIR / "NOTEBOOK26_fold_rule_sensitivity_validation_report.json"
global_validation_path = REPORT_DIR / f"NOTEBOOK26_fold_rule_sensitivity_validation_report_{RUN_ID}.json"

save_json(validation_path, validation_report)
save_json(global_validation_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)
manifest_path = REPORT_RUN_DIR / "NOTEBOOK26_file_manifest_SHA256.csv"
global_manifest_path = REPORT_DIR / f"NOTEBOOK26_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(global_manifest_path, index=False)

# ============================================================
# 11. Final summary
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 26 COMPLETE")
print("=" * 100)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("Input index:", input_index_path)
print("Probability paths:")
for k, v in probability_paths.items():
    print(" ", k, ":", v)

print("\nMain output tables:")
print("Table S38:", TABLE_RUN_DIR / "table_S38_fold_rule_sensitivity_design.csv")
print("Table S39:", TABLE_RUN_DIR / "table_S39_fold_rule_probability_sensitivity.csv")
print("Table S40:", TABLE_RUN_DIR / "table_S40_fold_rule_portfolio_sensitivity.csv")
print("Table S41:", TABLE_RUN_DIR / "table_S41_fold_rule_source_composition.csv")
print("Table S42:", TABLE_RUN_DIR / "table_S42_fold_rule_difference_vs_latest.csv")

print("\nDiagnostics:")
print("Probability map:", DIAG_DIR / "fold_rule_selected_probability_map.csv")
print("Returns:", RETURN_DIR / "fold_rule_sensitivity_returns.parquet")
print("Weights:", WEIGHT_DIR / "fold_rule_sensitivity_weights.parquet")
print("Optimizer diagnostics:", DIAG_DIR / "fold_rule_optimizer_diagnostics.csv")
print("Validation report:", validation_path)
print("Manifest:", manifest_path)

print("\nTable S39 preview:")
print(table_S39.round(6).to_string(index=False))

print("\nTable S40 preview:")
print(table_S40.round(6).to_string(index=False))

print("\nTable S42 preview:")
print(table_S42.round(6).to_string(index=False))

print("\nInterpretation helper:")
print(interpret_df.to_string(index=False) if not interpret_df.empty else "No interpretation rows.")

print("\nRecommended manuscript interpretation template:")
print(
    "Fold-rule sensitivity compares latest eligible fold, earliest eligible fold, and averaged eligible probabilities. "
    "If portfolio metrics and cash exposure are close across rules, the overlapping-fold retention rule does not materially "
    "affect the mechanism conclusion. If probability metrics differ but portfolio metrics remain similar, the result further "
    "supports the interpretation that the allocation layer is dominated by defensive cash exposure and constraints."
)
print("=" * 100)

Mounted at /content/drive
AURORA-TWETF Notebook 26: Fold-rule sensitivity
RUN_ID: 20260728_075553
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/fold_rule_sensitivity/run_20260728_075553
Evaluation window: 2024-11-27 to 2026-03-25

Loading modeling labels and ETF return panel
Label file: /content/drive/MyDrive/AURORA_TWETF/data/modeling/AURORA_TWETF_features_with_labels.parquet
Label shape: (1262, 2)
Label dates: 2021-01-06 to 2026-03-25
ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
ETF return shape: (1426, 4)
ETF return dates: 2021-01-01 to 2026-06-23
Base aligned evaluation dates: 319 2024-11-27 to 2026-03-25

Loading selected pre-deduplication probability files
Allocation input index: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv
Index shape: (2, 11)
Index columns: ['run_id', 'policy_name', 'target_col', 